# Importing Key Libraries

In [205]:
import pandas as pd
import numpy as np

import pickle
import os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# Dataset preparation

In [206]:
df = pd.read_csv('../data/CreditScoringCleaned.csv')

In [207]:
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,ok,9,rent,60,30,married,no,freelance,73,129.0,0.0,0.0,800,846
1,ok,17,rent,60,58,widow,no,fixed,48,131.0,0.0,0.0,1000,1658
2,default,10,owner,36,46,married,yes,freelance,90,200.0,3000.0,0.0,2000,2985
3,ok,0,rent,60,24,single,no,fixed,63,182.0,2500.0,0.0,900,1325
4,ok,0,rent,36,26,single,no,fixed,46,107.0,0.0,0.0,310,910


In [208]:
df.isnull().sum()

status        0
seniority     0
home          0
time          0
age           0
marital       0
records       0
job           0
expenses      0
income       34
assets       47
debt         18
amount        0
price         0
dtype: int64

In [209]:
df = df.fillna(0)

In [210]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4454.0,4454.0,4454.0,4454.0,4454.0,4454.0,4454.0,4454.0,4454.0
mean,8.0,46.0,37.0,56.0,130.0,5347.0,342.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,87.0,11526.0,1244.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1117.0
50%,5.0,48.0,36.0,51.0,119.0,3000.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,164.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,959.0,300000.0,30000.0,5000.0,11140.0


In [211]:
df.isnull().sum()

status       0
seniority    0
home         0
time         0
age          0
marital      0
records      0
job          0
expenses     0
income       0
assets       0
debt         0
amount       0
price        0
dtype: int64

### Splitting the dataset into train, validation, and test

In [212]:
from sklearn.model_selection import train_test_split

df_train_full, df_test = train_test_split(df, test_size = 0.2, random_state = 11)
df_train, df_val = train_test_split(df_train_full, test_size = 0.25, random_state=11)

In [213]:
len(df_train), len(df_val), len(df_test)

(2672, 891, 891)

In [214]:
y_train = (df_train.status == 'default').values
y_train

array([ True,  True, False, ..., False, False, False])

In [215]:
len(y_train)

2672

In [216]:
y_val = (df_val.status == 'default').values
y_train

array([ True,  True, False, ..., False, False, False])

In [217]:
len(y_val)

891

In [218]:
y_test = (df_test.status == 'default').values
y_test

array([ True, False, False, False,  True, False, False, False, False,
       False, False, False, False,  True,  True, False,  True,  True,
       False,  True, False, False,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False,  True,
       False, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False, False,  True,  True, False,
        True, False, False, False,  True,  True, False,  True, False,
       False, False, False,  True,  True, False, False,  True, False,
       False, False, False, False, False, False, False, False, False,
       False, False,  True, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False,  True,  True, False,  True, False,  True, False,  True,
        True,  True, False, False, False, False, False, False, False,
       False, False,

In [219]:
len(y_test)

891

In [220]:
del df_train['status']
del df_val['status']
del df_test['status']

In [221]:
# XGBoost has built-in support for missing values (NaN).

'''
df_train = df_train.fillna(0)
df_val = df_val.fillna(0)
df_test = df_test.fillna(0)
'''

'\ndf_train = df_train.fillna(0)\ndf_val = df_val.fillna(0)\ndf_test = df_test.fillna(0)\n'

In [222]:
df_train

,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
951,10,owner,36,36,married,no,freelance,75,0.0,10000.0,0.0,1000,1400
688,6,parents,48,32,single,yes,fixed,35,85.0,0.0,0.0,1100,1330
2233,1,parents,48,40,married,no,fixed,75,121.0,0.0,0.0,1320,1600
3304,1,parents,48,23,single,no,parttime,35,72.0,0.0,0.0,1078,1079
2271,5,owner,36,46,married,no,freelance,60,100.0,4000.0,0.0,1100,1897
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,18,private,36,45,married,no,fixed,45,220.0,20000.0,0.0,800,1600
1784,7,private,60,29,married,no,fixed,60,51.0,3500.0,500.0,1000,1290
808,1,parents,24,19,single,no,fixed,35,28.0,0.0,0.0,400,600
1857,15,owner,48,43,married,no,freelance,60,100.0,18000.0,0.0,2500,2976


In [223]:
dict_train = df_train.to_dict(orient='records')
dict_val = df_val.to_dict(orient='records')
dict_test = df_test.to_dict(orient='records')

In [224]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse = False)
X_train = dv.fit_transform(dict_train)
X_val = dv.transform(dict_val)
X_test = dv.transform(dict_test)

In [225]:
X_train

array([[3.60e+01, 1.00e+03, 1.00e+04, ..., 0.00e+00, 1.00e+01, 3.60e+01],
       [3.20e+01, 1.10e+03, 0.00e+00, ..., 1.00e+00, 6.00e+00, 4.80e+01],
       [4.00e+01, 1.32e+03, 0.00e+00, ..., 0.00e+00, 1.00e+00, 4.80e+01],
       ...,
       [1.90e+01, 4.00e+02, 0.00e+00, ..., 0.00e+00, 1.00e+00, 2.40e+01],
       [4.30e+01, 2.50e+03, 1.80e+04, ..., 0.00e+00, 1.50e+01, 4.80e+01],
       [2.70e+01, 4.50e+02, 5.00e+03, ..., 1.00e+00, 1.20e+01, 4.80e+01]])

In [226]:
X_val

array([[3.10e+01, 5.50e+02, 0.00e+00, ..., 0.00e+00, 6.00e+00, 3.60e+01],
       [3.80e+01, 1.00e+03, 0.00e+00, ..., 0.00e+00, 1.80e+01, 6.00e+01],
       [4.00e+01, 7.00e+02, 0.00e+00, ..., 1.00e+00, 1.70e+01, 2.40e+01],
       ...,
       [3.60e+01, 3.90e+03, 2.90e+04, ..., 1.00e+00, 2.00e+00, 6.00e+01],
       [2.50e+01, 3.00e+02, 0.00e+00, ..., 1.00e+00, 3.00e+00, 2.40e+01],
       [3.20e+01, 1.55e+03, 6.00e+03, ..., 0.00e+00, 1.50e+01, 6.00e+01]])

In [227]:
X_test

array([[2.60e+01, 8.00e+02, 6.00e+04, ..., 0.00e+00, 3.00e+00, 3.60e+01],
       [2.80e+01, 2.25e+03, 1.80e+01, ..., 0.00e+00, 1.00e+01, 6.00e+01],
       [4.10e+01, 1.15e+03, 0.00e+00, ..., 0.00e+00, 1.40e+01, 6.00e+01],
       ...,
       [2.80e+01, 6.00e+02, 6.00e+03, ..., 0.00e+00, 0.00e+00, 2.40e+01],
       [3.00e+01, 1.22e+03, 4.00e+03, ..., 1.00e+00, 8.00e+00, 3.60e+01],
       [4.90e+01, 8.50e+02, 1.50e+04, ..., 0.00e+00, 4.00e+00, 2.40e+01]])

In [228]:
dv.feature_names_

['age',
 'amount',
 'assets',
 'debt',
 'expenses',
 'home=ignore',
 'home=other',
 'home=owner',
 'home=parents',
 'home=private',
 'home=rent',
 'home=unknown',
 'income',
 'job=fixed',
 'job=freelance',
 'job=others',
 'job=parttime',
 'job=unknown',
 'marital=divorced',
 'marital=married',
 'marital=separated',
 'marital=single',
 'marital=unknown',
 'marital=widow',
 'price',
 'records=no',
 'records=yes',
 'seniority',
 'time']

# Gradient Boosting

In [229]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=dv.feature_names_)

In [230]:
xgb_params = {
'eta': 0.1,
'max_depth': 3,
'min_child_weight': 1,
'objective': 'binary:logistic',
'eval_metric': 'auc',
'nthread': 8,
'seed': 1,
'silent': 1
}

In [231]:
num_trees = 160
model = xgb.train(xgb_params, dtrain, num_boost_round=num_trees)

In [232]:
y_pred_xgb = model.predict(dval)
roc_auc_score(y_val, y_pred_xgb)

0.833937783051997

In [233]:
y_pred_xgb = model.predict(dtest)
roc_auc_score(y_test, y_pred_xgb)

0.8244037437075411

In [234]:
ROOT_DIR = Path.cwd().parent

file_path = os.path.join(ROOT_DIR, 'artifacts', 'xgboost-model.bin') 

os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'wb') as f_out:
    pickle.dump((dv, model),f_out)

In [167]:
ROOT_DIR = Path.cwd().parent

file_path = os.path.join(ROOT_DIR, 'artifacts', 'xgboost-model.bin') 

os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [204]:
y_pred_xgb = model.predict(dval)
roc_auc_score(y_val, y_pred_xgb)

0.833937783051997

In [238]:
new_application = {
    "seniority": 5,
    "home": "owner",
    "time": 48,
    "age": 35,
    "marital": "married",
    "records": "no",
    "job": "fixed",
    "expenses": 80,
    "income": 150,
    "assets": 5000,
    "debt": 0,
    "amount": 1200,
    "price": 1600
}

X_custom = dv.transform([new_application])
dmatrix_custom = xgb.DMatrix(X_custom, feature_names=dv.feature_names_)
prob = model.predict(dmatrix_custom)[0]
pred = 'There is chance of default: Loan should not be disbursed' if prob >= 0.3 else 'There is low chance of default: Loan can be disbursed'

print(f"Default probability: {prob:.4f}")
print(f"Classification: {pred}")

Default probability: 0.0637
Classification: There is low chance of default: Loan can be disbursed
